## Load files

In [1]:
import json
import pandas as pd


## Annotating Paragraph

The annotation process made me realize that stance labeling is usually easier than relevance labeling, because many perspectives clearly support or oppose a claim, while relevance often sits on a spectrum and requires more judgment. I learned how important it is to separate agreement from topical fit. A perspective can be strongly worded and still be only somewhat relevant if it addresses a side issue rather than the core issue. It didn’t feel hard, just tedious. It was a little challenging when wording became vague and broad. A next iteration of the task would benefit from clearer annotation guidelines, especially with more edge-case examples showing the difference between “directly relevant” and “somewhat relevant”.  Overall, the exercise changed my perception of annotation by showing me that it is not just mechanical labeling; it is a careful interpretive process where good definitions and consistency matter as much as the labels themselves.

In [2]:
with open("perspectrum_with_answers_v1.0.json", "r", encoding="utf-8") as f:
    perspectrum_data = json.load(f)

with open("perspective_pool_v1.0.json", "r", encoding="utf-8") as f:
    perspective_pool_data = json.load(f)
    
annotated_df = pd.read_csv("annotated_sample_100.csv")
print(annotated_df.head())

   cid                                         claim_text    pid  \
0  942       Children should do part time and summer work  26954   
1  395  The United Nations has a responsibility to pro...   2940   
2  504         Encourage fewer people to go to university  24893   
3  564  The United States should continue Its use of d...   4191   
4  176                    Make all museums free of charge   1309   

                                    perspective_text stance_annotation  \
0  Children should work because a job provides sa...           support   
1  Blanket commitment creates a slippery slope of...            oppose   
2  Many people with college degrees find themselv...           support   
3  Drone strikes are subject to a strict review p...           support   
4  If state-funded, there is little incentive to ...            oppose   

  relevance_annotation  
0    directly relevant  
1    directly relevant  
2    directly relevant  
3    directly relevant  
4    directly relevan

## Setup
I set up a Python virtual environment on Great Lakes, installed the required packages, copied the dataset and annotation files to the cluster, and verified that PyTorch detected a GPU before training.

## Part 2


In [3]:
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

/Users/lamontenunn/Desktop/SI630/Homework3/.venv-1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
SEED = 440
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PERSPECTRUM_FILE = "perspectrum_with_answers_v1.0.json"
PERSPECTIVE_POOL_FILE = "perspective_pool_v1.0.json"
MODEL_NAME = "roberta-base"

## Load files

In [5]:
with open("perspectrum_with_answers_v1.0.json", "r", encoding="utf-8") as f:
    perspectrum_data = json.load(f)

with open("perspective_pool_v1.0.json", "r", encoding="utf-8") as f:
    perspective_pool_data = json.load(f)
    
    
print("Number of claims:", len(perspectrum_data))
print("Number of perspective pool items:", len(perspective_pool_data))

Number of claims: 907
Number of perspective pool items: 11112


In [6]:
pid_to_text = {}
for item in perspective_pool_data:
    pid_to_text[item["pId"]] = item["text"]

print("Lookup size:", len(pid_to_text))

Lookup size: 11112


In [7]:
def map_labels(stance_label_5):
    label = stance_label_5.strip().lower()

    if label == "support":
        return 1, 1
    elif label == "undermine":
        return 0, 1
    elif label == "mildly support":
        return 1, 0
    elif label == "mildly undermine":
        return 0, 0
    else:
        return None

In [8]:
rows = []

for claim in perspectrum_data:
    cid = claim["cId"]
    claim_text = claim["text"]

    for perspective_group in claim["perspectives"]:
        pids = perspective_group["pids"]
        stance_label_5 = perspective_group["stance_label_5"]
        voter_counts = perspective_group["voter_counts"]

        # Drop not a valid perspective
        # identify these by nonzero last entry of voter_counts
        if len(voter_counts) > 4 and voter_counts[-1] > 0:
            continue

        # Drop no-majority cases
        if stance_label_5.strip().upper() == "NO MAJORITY LABEL":
            continue

        mapped = map_labels(stance_label_5)
        if mapped is None:
            continue

        stance_support, relevance_direct = mapped

        if not pids:
            continue

        # Use all pids in the group for training so we do not throw away usable text examples
        for pid in pids:
            perspective_text = pid_to_text.get(pid)

            if perspective_text is None:
                continue

            rows.append({
                "cid": cid,
                "claim_text": claim_text,
                "pid": pid,
                "perspective_text": perspective_text,
                "stance_support": stance_support,
                "relevance_direct": relevance_direct
            })

df = pd.DataFrame(rows).drop_duplicates()

print("Training rows after cleaning:", len(df))
print(df.head())

Training rows after cleaning: 10062
   cid                           claim_text    pid  \
0  499  Vaccination must be made compulsory   3695   
1  499  Vaccination must be made compulsory  24076   
2  499  Vaccination must be made compulsory  24077   
3  499  Vaccination must be made compulsory   3698   
4  499  Vaccination must be made compulsory  24082   

                                    perspective_text  stance_support  \
0   It is the state’s duty to protect its community                1   
1           The state must keep it's community safe.               1   
2  The safety of the community is the state's pri...               1   
3  Compulsory vaccination violates the individual...               0   
4  Individuals have the right to refuse vaccinati...               0   

   relevance_direct  
0                 1  
1                 1  
2                 1  
3                 1  
4                 1  


In [9]:
df["labels"] = df.apply(
    lambda row: [float(row["stance_support"]), float(row["relevance_direct"])],
    axis=1
)

df[["claim_text", "perspective_text", "stance_support", "relevance_direct"]].head()

,claim_text,perspective_text,stance_support,relevance_direct
0,Vaccination must be made compulsory,It is the state’s duty to protect its community,1,1
1,Vaccination must be made compulsory,The state must keep it's community safe.,1,1
2,Vaccination must be made compulsory,The safety of the community is the state's pri...,1,1
3,Vaccination must be made compulsory,Compulsory vaccination violates the individual...,0,1
4,Vaccination must be made compulsory,Individuals have the right to refuse vaccinati...,0,1


## Train / Dev Split

I stratify on the joint stance-relevance label so the dev set has a similar label mix to the training set.

In [10]:
train_df, dev_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["stance_support"].astype(str) + "_" + df["relevance_direct"].astype(str)
)

train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Dev size:", len(dev_df))

print("\nTrain label distribution:")
print(train_df[["stance_support", "relevance_direct"]].value_counts())

print("\nDev label distribution:")
print(dev_df[["stance_support", "relevance_direct"]].value_counts())

Train size: 8049
Dev size: 2013

Train label distribution:
stance_support  relevance_direct
1               1                   4251
0               1                   3798
Name: count, dtype: int64

Dev label distribution:
stance_support  relevance_direct
1               1                   1063
0               1                    950
Name: count, dtype: int64


## Convert to Hugging Face Datasets

The model input is the claim text paired with the perspective text. The label vector has two entries:
1. `stance_support` where 1 = support and 0 = oppose
2. `relevance_direct` where 1 = directly relevant and 0 = somewhat relevant

In [11]:
train_dataset = Dataset.from_pandas(
    train_df[["claim_text", "perspective_text", "labels"]],
    preserve_index=False
)

dev_dataset = Dataset.from_pandas(
    dev_df[["claim_text", "perspective_text", "labels"]],
    preserve_index=False
)

print(train_dataset)
print(dev_dataset)

Dataset({
    features: ['claim_text', 'perspective_text', 'labels'],
    num_rows: 8049
})
Dataset({
    features: ['claim_text', 'perspective_text', 'labels'],
    num_rows: 2013
})


## Tokenization

I use RoBERTa with paired inputs `(claim_text, perspective_text)` and truncate sequences to length 256.

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["claim_text"],
        examples["perspective_text"],
        truncation=True,
        padding=False,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
dev_dataset = dev_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 2013/2013 [00:00<00:00, 21679.19 examples/s]


In [13]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Model and Metrics

This is a multilabel setup with two jointly predicted binary labels, so I use `problem_type='multi_label_classification'` and threshold sigmoid outputs at 0.5 during evaluation.

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="multi_label_classification"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    subset_acc = accuracy_score(labels, preds)

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "subset_accuracy": subset_acc
    }

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4784.11it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training

For my final Problem 6 run, I will switch `report_to` from `'none'` to `'wandb'` after logging in with Weights & Biases.

In [20]:
training_args = TrainingArguments(
    output_dir="outputs/roberta_hw3",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=2,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [ ]:
trainer.train()

In [ ]:
dev_results = trainer.evaluate()
dev_results